## 05 - Monitoring and Model Registry

This notebook is a placeholder for setting up:

1. Model registration with SageMaker Model Registry  
2. Model and data monitoring (drift detection, baseline checks)


## Update Model With `inference.py` Need for Batch Processing

In [1]:
import os
import tarfile
from sagemaker import Session

# Paths
model_artifact = "model/model.joblib"
inference_script = "inference.py"
tarball_path = "registry/model.tar.gz"

# Ensure folders
os.makedirs("registry", exist_ok=True)

# Create model.tar.gz containing model.joblib and inference.py
with tarfile.open(tarball_path, "w:gz") as tar:
    tar.add(model_artifact, arcname="model.joblib")
    tar.add(inference_script, arcname="inference.py")

print(f"Packaged model and inference script into: {tarball_path}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Packaged model and inference script into: registry/model.tar.gz


## Sagemaker Setup

In [2]:
# Initialize session and define S3 path

session = Session()
bucket = session.default_bucket()

model_s3_uri = session.upload_data(
    path=tarball_path,
    bucket=bucket,
    key_prefix="diabetes/registry"
)

print("Uploaded model to:", model_s3_uri)


Uploaded model to: s3://sagemaker-us-east-1-380537322556/diabetes/registry/model.tar.gz


## Create Model Package Group

In [3]:
import boto3
import sagemaker
from sagemaker import image_uris

model_package_group_name = "ReadmissionModelGroup"

# Get the Scikit-learn container URI for current region
region = session.boto_region_name
sklearn_uri = image_uris.retrieve(framework="sklearn", region=region, version="1.0-1")

sm_client = boto3.client("sagemaker")

# Check if model package group already exists
existing_groups = sm_client.list_model_package_groups(
    NameContains=model_package_group_name
).get("ModelPackageGroupSummaryList", [])

if any(group["ModelPackageGroupName"] == model_package_group_name for group in existing_groups):
    print("Model package group already exists:", model_package_group_name)
else:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Logistic Regression for hospital readmission prediction"
    )
    print("Created model package group:", model_package_group_name)


Model package group already exists: ReadmissionModelGroup


## Register Model

In [4]:
# Register the model version
response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Logistic Regression model (C=0.01, class_weight=balanced)",
    InferenceSpecification={
        "Containers": [{
            "Image": sklearn_uri,
            "ModelDataUrl": model_s3_uri
        }],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    },
    ModelApprovalStatus="PendingManualApproval"
)

print("Registered model version in:", model_package_group_name)
print("Model Package ARN:", response["ModelPackageArn"])


Registered model version in: ReadmissionModelGroup
Model Package ARN: arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/18


# Model Monitoring

## Sagemaker Setup

In [47]:
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model_monitor import DataCaptureConfig, DefaultModelMonitor
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.session import Session
import os

# Set up sessions and role
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_session = Session(boto_session=boto_session)
role = get_execution_role()

bucket = sagemaker_session.default_bucket()


## Initialize Monitor

In [2]:
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    base_job_name="readmission-monitor",
    sagemaker_session=sagemaker_session
)


## Create Baseline

In [3]:
import pandas as pd

# Load your validation dataset (with features only)
X_val = pd.read_pickle("data/X_val.pkl")  # or read your CSV directly
X_val.to_csv("data/X_val_for_baseline.csv", index=False, header=False)


In [5]:
import contextlib
from sagemaker.model_monitor import DatasetFormat

# Create a logs folder if it doesn’t exist
os.makedirs("log", exist_ok=True)

with open("log/baseline_job_log.txt", "w") as f, contextlib.redirect_stdout(f):
    baseline_job = monitor.suggest_baseline(
        baseline_dataset="data/X_val_for_baseline.csv",
        dataset_format=DatasetFormat.csv(header=False),
        output_s3_uri=f"s3://{bucket}/monitoring/baseline",
        wait=True
    )

print("Baseline job complete. Statistics and constraints saved to S3.")
print("Details logged in: log/baseline_job_log.txt")


INFO:sagemaker:Creating processing-job with name readmission-monitor-2025-06-24-17-20-56-843


Baseline job complete. Statistics and constraints saved to S3.
Details logged in: log/baseline_job_log.txt


In [18]:
# Check outputs

import boto3

s3 = boto3.client("s3")
baseline_prefix = "monitoring/baseline/"

objects = s3.list_objects(Bucket=bucket, Prefix=baseline_prefix)
print("Baseline outputs in S3:")
for obj in objects.get("Contents", []):
    print("-", obj["Key"])


Baseline outputs in S3:
- monitoring/baseline/constraints.json
- monitoring/baseline/statistics.json
